# 01. Environment Setup & Data Acquisition Pipeline

This notebook configures the required runtime environment (ISCE2, MintPy, GDAL) and handles automated retrieval of 124 Sentinel-1 Single Look Complex (SLC) scenes and Precise Orbit Ephemerides (POD) for the Sayung, Demak InSAR processing stack.

### Pipeline Outline:
1. Environment Setup (Python 3.11 with CondaColab, ISCE2, MintPy)
2. System Path & Environment Variables Configuration
3. NASA Earthdata Credential Setup (`.netrc`)
4. Querying & Batch Downloading Sentinel-1 SLC Data via `asf_search`
5. Orbit File Retrieval (`fetchOrbit.py`)

## 1. Environment Setup
ISCE2 version 2.6.3 prebuilt binaries on conda-forge require a Python 3.11 environment. CondaColab is initialized to bootstrap Miniforge into Google Colab.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

## 2. Resolving Dependencies and Installing Packages
Remove Colab's default conda pinning to install Python 3.11 along with ISCE2, MintPy, and spatial libraries.

In [ ]:
import os
import sys

!rm -f /usr/local/conda-meta/pinned
!mamba install -c conda-forge isce2=2.6.3 mintpy python=3.11 pyresample gdal -y

## 3. Configuring ISCE2 & System PATH
Define `ISCE_HOME`, update Python path, and add ISCE applications to the system PATH.

In [ ]:
site_packages = "/usr/local/lib/python3.11/site-packages"
isce_home = os.path.join(site_packages, "isce")

os.environ["ISCE_HOME"] = isce_home
os.environ["PYTHONPATH"] = f"{site_packages}:{os.environ.get('PYTHONPATH', '')}"
os.environ["PATH"] = (
    f"{isce_home}/bin:{isce_home}/applications:/usr/local/share/isce2/topsStack:{os.environ['PATH']}"
)
sys.path.append(site_packages)

# Verification
import isce
import mintpy

print("ISCE2 installation verified at:", isce_home)
!smallbaselineApp.py --version

## 4. NASA Earthdata Authentication & Directory Setup
Creates a secure `.netrc` file required for accessing NASA ASF DAAC repositories and prepares workspace directories.

In [ ]:
import getpass
import os
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/Thesis_Hands_On"
RAW_DATA_DIR = os.path.join(BASE_DIR, "raw_data")
ORBIT_DIR = os.path.join(RAW_DATA_DIR, "orbits")

os.makedirs(ORBIT_DIR, exist_ok=True)

earthdata_user = input("Enter NASA Earthdata Username: ")
earthdata_pass = getpass.getpass("Enter NASA Earthdata Password: ")

netrc_content = f"""machine urs.earthdata.nasa.gov login {earthdata_user} password {earthdata_pass}
machine auig2.asf.alaska.edu login {earthdata_user} password {earthdata_pass}
"""

netrc_path = os.path.expanduser("~/.netrc")
with open(netrc_path, "w") as f:
  f.write(netrc_content)

os.chmod(netrc_path, 0o600)
print("Configured .netrc successfully.")

## 5. Query and Download Sentinel-1 SLC Scenes
Queries the Alaska Satellite Facility (ASF) archive for Sentinel-1 IW SLC scenes covering Path 76, Frame 613 from January 2021 to April 2026. Duplicate scene acquisition dates are removed to isolate exactly 124 unique temporal observations.

In [ ]:
!pip install -q asf_search
import os
import time
import asf_search as asf

aoi_wkt = (
    "POLYGON((110.39 -7.005, 110.525 -7.005, 110.525 -6.87, 110.39 -6.87, 110.39"
    " -7.005))"
)

search_results = asf.geo_search(
    platform="SENTINEL-1",
    intersectsWith=aoi_wkt,
    start="2021-01-01T00:00:00Z",
    end="2026-04-30T23:59:59Z",
    processingLevel="SLC",
    beamMode="IW",
    flightDirection="DESCENDING",
    relativeOrbit=76,
    frame=613,
    polarization="VV+VH",
)

unique_scenes = {}
for scene in search_results:
  date_str = scene.properties["startTime"][:10]
  if date_str not in unique_scenes:
    unique_scenes[date_str] = scene

scenes_list = list(unique_scenes.values())
print(f"Total unique scenes found: {len(scenes_list)}")

for idx, scene in enumerate(scenes_list, start=1):
  filename = scene.properties["fileName"]
  target_file = os.path.join(RAW_DATA_DIR, filename)

  if os.path.exists(target_file):
    print(f"[{idx}/{len(scenes_list)}] Skipped (already exists): {filename}")
    continue

  success = False
  for attempt in range(1, 4):
    try:
      print(
          f"[{idx}/{len(scenes_list)}] Downloading: {filename} (Attempt"
          f" {attempt})..."
      )
      scene.download(path=RAW_DATA_DIR)
      success = True
      break
    except Exception as e:
      print(f"Download failed: {e}. Retrying in 5 seconds...")
      time.sleep(5)

  if not success:
    print(f"ERROR: Unable to download {filename} after 3 attempts.")

## 6. Precise Orbit Ephemerides (POD) Retrieval
Fetches precise orbit files (.EOF) corresponding to each Sentinel-1 scene using topsStack's `fetchOrbit.py`.

In [ ]:
import glob
import os
import subprocess

fetch_orbit_script = "/usr/local/share/isce2/topsStack/fetchOrbit.py"

cmd = ["python", fetch_orbit_script, "-i", RAW_DATA_DIR, "-o", ORBIT_DIR]
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
  print("Batch execution encountered an error, falling back to per-file query:")
  zip_files = glob.glob(os.path.join(RAW_DATA_DIR, "*.zip"))
  for z in zip_files:
    subprocess.run(["python", fetch_orbit_script, "-i", z, "-o", ORBIT_DIR])

eof_files = glob.glob(os.path.join(ORBIT_DIR, "*.EOF"))
print(f"Total precise orbit files downloaded: {len(eof_files)}")